In [1]:
import pandas as pd
import nltk
import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re

In [2]:
# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Sean\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
# 1. Read the data (using only the 'text' column)
# Replace 'news_dataset.csv' with the correct path to your file
try:
    df = pd.read_csv(r'C:\Users\Sean\Desktop\news_dataset.csv')
    df = df[['text']]
except FileNotFoundError:
    print("Error: news_dataset.csv not found. Please ensure the file is in the same directory.")


In [6]:
# 2. Text Pre-processing
# Remove null values
df = df.dropna(subset=['text'])

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert to lowercase and remove non-alphabetical characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords, perform Stemming and Lemmatization
    processed_tokens = [
        lemmatizer.lemmatize(stemmer.stem(w)) 
        for w in tokens if w not in stop_words and len(w) > 3
    ]
    return processed_tokens

df['processed_text'] = df['text'].apply(preprocess_text)


In [7]:
# 3. Perform LDA using Gensim
# Create Dictionary
id2word = corpora.Dictionary(df['processed_text'])

# Create Corpus (Term Document Frequency)
texts = df['processed_text']
corpus = [id2word.doc2bow(text) for text in texts]

# Set the number of topics to 4 as required
num_topics = 4

# Build LDA model
lda_model = gensim.models.ldamodel.LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=num_topics, 
    random_state=100,
    update_every=1,
    chunksize=100,
    passes=10,
    alpha='auto',
    per_word_topics=True
)


In [8]:
# Print the 4 topics discovered
print("\n--- Discovered Topics ---")
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")

# 4. Evaluate the LDA model using Coherence score
coherence_model_lda = CoherenceModel(
    model=lda_model, 
    texts=df['processed_text'], 
    dictionary=id2word, 
    coherence='c_v'
)
coherence_score = coherence_model_lda.get_coherence()
print(f'\nCoherence Score: {coherence_score}')



--- Discovered Topics ---
Topic 0: 0.080*"chip" + 0.031*"escrow" + 0.025*"devic" + 0.015*"serial" + 0.011*"phone" + 0.011*"disk" + 0.010*"instal" + 0.009*"block" + 0.008*"manufactur" + 0.008*"cellular"
Topic 1: 0.032*"encrypt" + 0.013*"anonym" + 0.013*"use" + 0.012*"file" + 0.012*"system" + 0.012*"inform" + 0.011*"key" + 0.011*"privaci" + 0.011*"comput" + 0.011*"messag"
Topic 2: 0.009*"would" + 0.008*"peopl" + 0.005*"know" + 0.005*"like" + 0.005*"dont" + 0.005*"govern" + 0.005*"think" + 0.005*"time" + 0.005*"right" + 0.004*"make"
Topic 3: 0.003*"trinomi" + 0.003*"leaf" + 0.002*"sweden" + 0.002*"boomer" + 0.002*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.002*"calgari" + 0.002*"pen" + 0.002*"scorer" + 0.002*"drake" + 0.002*"winnipeg"

Coherence Score: 0.5625340578146992


--- Discovered Topics ---
Topic 0: 0.080*"chip" + 0.031*"escrow" + 0.025*"devic" + 0.015*"serial" + 0.011*"phone" + 0.011*"disk" + 0.010*"instal" + 0.009*"block" + 0.008*"manufactur" + 0.008*"cellular"
Topic 1: 0.032*"encrypt" + 0.013*"anonym" + 0.013*"use" + 0.012*"file" + 0.012*"system" + 0.012*"inform" + 0.011*"key" + 0.011*"privaci" + 0.011*"comput" + 0.011*"messag"
Topic 2: 0.009*"would" + 0.008*"peopl" + 0.005*"know" + 0.005*"like" + 0.005*"dont" + 0.005*"govern" + 0.005*"think" + 0.005*"time" + 0.005*"right" + 0.004*"make"
Topic 3: 0.003*"trinomi" + 0.003*"leaf" + 0.002*"sweden" + 0.002*"boomer" + 0.002*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.002*"calgari" + 0.002*"pen" + 0.002*"scorer" + 0.002*"drake" + 0.002*"winnipeg"

Coherence Score: 0.5625340578146992